In [0]:
%pip install h3 shapely --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./00_config

In [0]:
RES = 7            # ~5 km² por celda
RIO_BUFFER = 0.02  # grados (~2 km) alrededor de ríos principales

# Umbrales de la regla final (elegidos con la tabla de sensibilidad)
DIST_MIN_KM = 5
BUQUES_MAX  = 3
POS_MAX     = 100

tbl_ref = T("silver", "ref_h3_superficie")

In [0]:
import os, urllib.request

ref_dir = V("ref")
base = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/"
archivos = ["ne_10m_land.geojson", "ne_10m_lakes.geojson", "ne_10m_rivers_lake_centerlines.geojson"]

for a in archivos:
    p = f"{ref_dir}/{a}"
    if not os.path.exists(p):
        urllib.request.urlretrieve(base + a, p)
    print(a, round(os.path.getsize(p) / 1e6, 1), "MB")

In [0]:
from pyspark.sql import functions as F

celdas = (spark.table(T("bronze", "ais_raw"))
          .select(F.expr(f"h3_longlatash3string(CAST(LON AS DOUBLE), CAST(LAT AS DOUBLE), {RES})").alias("h3"), "MMSI")
          .groupBy("h3")
          .agg(F.count("*").alias("posiciones"), F.countDistinct("MMSI").alias("buques")))
pdf = celdas.toPandas()
print("celdas observadas:", len(pdf))

In [0]:
import json, h3, numpy as np, pandas as pd
from shapely.geometry import shape, Polygon, Point
from shapely import STRtree

def cargar(nombre):
    with open(f"{ref_dir}/{nombre}") as fh:
        return [shape(f["geometry"]) for f in json.load(fh)["features"] if f["geometry"]]

land, lakes, rivers = (cargar(a) for a in archivos)
t_land, t_lake, t_riv = STRtree(land), STRtree(lakes), STRtree(rivers)

def poly_celda(c):
    pts = [(lng, lat) for lat, lng in h3.cell_to_boundary(c)]
    lons = [p[0] for p in pts]
    return None if max(lons) - min(lons) > 180 else Polygon(pts)   # antimeridiano → agua

polys = [poly_celda(c) for c in pdf.h3]
idx = np.array([i for i, p in enumerate(polys) if p is not None])
g = [polys[i] for i in idx]

def marca(pares):
    m = np.zeros(len(g), bool)
    m[np.unique(pares[0])] = True
    return m

dentro_tierra = marca(t_land.query(g, predicate="within"))
toca_lago     = marca(t_lake.query(g, predicate="intersects"))
cerca_rio     = marca(t_riv.query(g, predicate="dwithin", distance=RIO_BUFFER))

base_tierra = np.zeros(len(pdf), bool)
base_tierra[idx] = dentro_tierra & ~toca_lago & ~cerca_rio
pdf["es_tierra_base"] = base_tierra
pdf[["lat", "lon"]] = [h3.cell_to_latlng(c) for c in pdf.h3]

print(f"dentro de tierra: {dentro_tierra.sum():,} | descartadas por lago: {(dentro_tierra & toca_lago).sum():,} "
      f"| descartadas por río: {(dentro_tierra & ~toca_lago & cerca_rio).sum():,} | base tierra: {base_tierra.sum():,}")

In [0]:
agua = STRtree([p.boundary for p in land] + lakes + rivers)

cand = np.where(base_tierra)[0]
pts = [Point(pdf.lon.iat[i], pdf.lat.iat[i]) for i in cand]
_, dist = agua.query_nearest(pts, return_distance=True, all_matches=False)
pdf["dist_agua_km"] = np.nan
pdf.loc[pdf.index[cand], "dist_agua_km"] = dist * 111   # grados → km (aprox.)
print("distancias calculadas:", len(cand))

In [0]:
tot = pdf.posiciones.sum()
filas = []
for dmin in [2, 5, 10]:
    for bmax in [3, 1]:
        for pmax in [10**9, 1000, 100]:
            m = pdf.es_tierra_base & (pdf.dist_agua_km >= dmin) & (pdf.buques <= bmax) & (pdf.posiciones <= pmax)
            filas.append({"dist_min_km": dmin, "buques_max": bmax, "pos_max": "∞" if pmax == 10**9 else pmax,
                          "celdas": int(m.sum()), "posiciones": int(pdf.posiciones[m].sum()),
                          "pct": round(100 * pdf.posiciones[m].sum() / tot, 4)})
print(pd.DataFrame(filas).to_string(index=False))

In [0]:
pdf["es_tierra"] = (pdf.es_tierra_base & (pdf.dist_agua_km >= DIST_MIN_KM)
                    & (pdf.buques <= BUQUES_MAX) & (pdf.posiciones <= POS_MAX))
pdf["resolucion"] = RES
pdf["regla"] = (f"NE10m land sin lagos, >{RIO_BUFFER}deg de rios, >={DIST_MIN_KM}km de agua, "
                f"<={BUQUES_MAX} buques, <={POS_MAX} posiciones")

cols_out = ["h3", "resolucion", "es_tierra", "es_tierra_base", "lat", "lon", "dist_agua_km", "posiciones", "buques", "regla"]
(spark.createDataFrame(pdf[cols_out])
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl_ref))

tot = pdf.posiciones.sum()
t = pdf[pdf.es_tierra]
print(f"celdas tierra: {len(t):,} | posiciones: {int(t.posiciones.sum()):,} de {int(tot):,} ({100*t.posiciones.sum()/tot:.3f}%)")
print("\nTop 15 celdas 'tierra' por posiciones:")
print(t.sort_values("posiciones", ascending=False).head(15)[["h3", "lat", "lon", "dist_agua_km", "posiciones", "buques"]].to_string(index=False))